In [0]:
!pip install dbldatagen
!pip install jmespath
!pip install pyspark
!pip install Faker

dbutils.library.restartPython()

In [0]:
import dbldatagen as dg
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, FloatType, StringType, TimestampType, 
    StructField, BooleanType, StructType, ArrayType, DecimalType
)

In [0]:
import yaml

# 1. Definir la ruta de tu archivo físico
ruta_archivo = '/Workspace/Users/jose.dataengineer@hotmail.com/Retailmax_data/data-generation/config.yaml'


try:
    # 2. Abrir y leer el archivo físico
    # Usamos encoding='utf-8' por buenas prácticas, especialmente al manejar español
    with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
        datos = yaml.safe_load(archivo)
        
    # 3. Extraer solo las llaves de primer nivel
    llaves_principales = list(datos.keys())
    print(f"Llaves de primer nivel: {llaves_principales}")
    
    # 4. Extraer las llaves de los elementos internos
    llaves_internas_tiendas = list(datos['tipos_tienda'][0].keys())
    print(f"Llaves internas de las tiendas: {llaves_internas_tiendas}")

except FileNotFoundError:
    print(f"Error: No se encontró el archivo '{ruta_archivo}'. Verifica que esté en la misma carpeta.")
except yaml.YAMLError as e:
    print(f"Error al analizar el archivo YAML: {e}")

In [0]:
import yaml
import dbldatagen as dg
from pyspark.sql.types import StringType


# Cargar YAML
with open('/Workspace/Users/jose.dataengineer@hotmail.com/Retailmax_data/data-generation/config1.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Extraer listas de valores y probabilidades desde el catálogo
tipos = [item['nombre'] for item in config['catalogos']['tipos_tienda']]
pesos_tipos = [item['probabilidad'] if item['probabilidad'] != 0 else 1 for item in config['catalogos']['tipos_tienda']]


# Aplicar al generador
generador = (
    dg.DataGenerator(spark, name="MSTR_TIENDAS", rows=100)
    .withColumn("id_tienda", "int", uniqueValues=100)
    .withColumn("tipo_tienda", StringType(), values=tipos, weights=pesos_tipos)
)

df_tiendas = generador.build()
display(df_tiendas)

In [0]:
!pip install dbldatagen
!pip install jmespath
!pip install pyspark
!pip install Faker
!pip install faker

dbutils.library.restartPython()

In [0]:
import yaml
import dbldatagen as dg
from dbldatagen import DataGenerator, fakerText, PyfuncText
from faker.providers import bank,company,internet,person

from pyspark.sql.types import StringType, IntegerType

from faker import Faker 

fake= Faker('es_CO')


# Cargar YAML

with open('/Workspace/Users/jose.dataengineer@hotmail.com/Retailmax_data/data-generation/config.yaml', 'r', encoding='utf-8') as f: config = yaml.safe_load(f)


# Extraer listas de valores y demanda desde el catálogo
tipos = [item['nombre'] for item in config['tipos_tienda']]
pesos_tipos = [item['demanda'] for item in config['tipos_tienda']]


# Aplicar al generador
generador = (
    dg.DataGenerator(spark, name="MSTR_TIENDAS", rows=150)
    .withColumn("id_tienda", IntegerType(), uniqueValues=150)
    .withColumn("tipo_tienda", StringType(), values=tipos, weights=pesos_tipos)
)

generador_articulos = (
    dg.DataGenerator(spark, name="MSTR_ARTICULOS", rows=5000)
    .withIdOutput("art_id")
    .withColumn("cod_barra",StringType(), template=r"###########")
    .withColumn("nombre_columna", StringType(), expression))


In [0]:
import dbldatagen as dg
row_count = 1_000_000

data_spec = (
    dg.DataGenerator(sparkSession=spark, name="transactions", rows=row_count)
    .withIdOutput()
    .withColumn("user_id", IntegerType(), minValue=1, maxValue=1_000_000)
    .withColumn("transaction_amount", FloatType(), minValue=1.0, maxValue=5000.0, random=True)
    .withColumn("transaction_date", TimestampType(), begin="2022-01-01 00:00:00", end="2022-12-31 23:59:59")
    .withColumn("product_category", StringType(), values=["Electronics", "Books", "Clothing", "Home", "Toys", "Sports", "Automotive"], random=True)
    # Add a Boolean column
    .withColumn("is_returned", BooleanType(), expr="transaction_amount < 100 AND rand() < 0.05")
    # Add an Array column
    .withColumn("tags", ArrayType(StringType()), expr="array(product_category, substr(transaction_date, 0, 10))")
    # Add a Struct column
    .withColumn("shipping_address", StructType([
        StructField("street", StringType(), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("zip", StringType(), True)
    ]), expr="""
        named_struct(
            'street', concat(cast(rand() * 9999 as INT), ' Main St'),
            'city', element_at(array('New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'), cast(rand() * 5 + 1 as INT)),
            'state', element_at(array('NY', 'CA', 'IL', 'TX', 'AZ'), cast(rand() * 5 + 1 as INT)),
            'zip', concat(cast(rand() * 89999 + 10000 as INT))
        )
    """)
    # Add a Decimal column
    .withColumn("tax", DecimalType(5, 2), expr="transaction_amount * 0.08")
    # Add a Date column
    .withColumn("delivery_date", TimestampType(), expr="date_add(transaction_date, cast(rand() * 7 as INT))")
    # Add a Column with Skewed Data
    .withColumn("payment_method", StringType(), values=["Credit Card"] * 80 + ["PayPal"] * 15 + ["Bitcoin"] * 5, random=True)
    # Introduce Null Values
    .withColumn("coupon_code", StringType(), expr="CASE WHEN rand() < 0.2 THEN concat('SAVE', cast(rand() * 100 as INT)) ELSE NULL END")
    # Add a Column with Dependent Values
    .withColumn("loyalty_points", IntegerType(), expr="CASE WHEN user_id % 2 = 0 THEN cast(transaction_amount / 10 as INT) ELSE 0 END")
    # Add a Nested Array of Structs
    .withColumn("items", ArrayType(StructType([
        StructField("item_id", IntegerType(), True),
        StructField("quantity", IntegerType(), True),
        StructField("price", FloatType(), True)
    ])), expr="""
        array(
            named_struct('item_id', cast(rand() * 1000 as INT), 'quantity', cast(rand() * 5 + 1 as INT), 'price', rand() * 100),
            named_struct('item_id', cast(rand() * 1000 as INT), 'quantity', cast(rand() * 5 + 1 as INT), 'price', rand() * 100)
        )
    """)
    # Add a Geospatial Data Column
    .withColumn("location", StringType(), expr="concat(cast(rand() * 180 - 90 as STRING), ', ', cast(rand() * 360 - 180 as STRING))")
)

In [0]:
# Build the DataFrame
df = data_spec.build()

# Explore the Generated Data
display(df)